In [ ]:
import numpy as np
import pandas as pd
import networkx as nx
import matplotlib.pyplot as plt
import seaborn as sns
from tinyshift.association_mining import TransactionAnalyzer
rng = np.random.default_rng(42)
from typing import List
import kagglehub
from kagglehub import KaggleDatasetAdapter
from econml.dml import CausalForestDML
from lightgbm import LGBMRegressor

In [293]:
df_groceries = kagglehub.dataset_load(KaggleDatasetAdapter.PANDAS, "irfanasrullah/groceries", "groceries - groceries.csv")
df_groceries = df_groceries[df_groceries.columns[1:]].fillna("")
groceries = df_groceries.to_numpy()
groceries = [arr[np.where(arr != "")] for arr in groceries]

In [ ]:
class CycleDemand:

    def __init__(self):
        self.mask_ = None
        self.skus_ = None
        self.cross_elasticity_ = None

    def fit_mask_(self, transactions: List[str]):
        ta = TransactionAnalyzer().fit(transactions)
        skus = ta.columns_
        lift = ta.correlation_matrix(skus, skus, metric="lift")
        hypergeom_p = ta.correlation_matrix(skus, skus, metric="hypergeom")
        mask = (hypergeom_p < 0.05) & (lift > 1.2) # TODO: salvar em esparsa esse dado, ou salvando via pd.DataFrame bool é eficiente?
        return skus, mask

    def analyze_competitors(self, sku_target: str):
        competing_skus = self.mask_.loc[sku_target][self.mask_.loc[sku_target]].index.tolist()
        competing_skus = [s for s in competing_skus if s != sku_target]
        return competing_skus

    def fit_cross_elasticity_(self, df_panel: pd.DataFrame):
        # É possível avaliar os produtos competidores?

        Y = df_panel["log_sales_B"].values
        T = df_panel[["log_price_A"]].values  # Preço do produto canibal
        X = df_panel[["trend", "month", "is_holiday", "marketing_spend", "log_price_B"]].values

        dml_model = CausalForestDML(
            model_y=LGBMRegressor(n_estimators=100, max_depth=3),
            model_t=LGBMRegressor(n_estimators=100, max_depth=3),
            discrete_treatment=False,
            random_state=42
        )
        dml_model.fit(Y, T, X=X)
        self.cross_elasticity_ = dml_model.effect(X)

    def fit_mlforecast_(self, df_panel: pd.DataFrame):
        # Salva o modelo nixtla via mlforecast, com as possíveis variaveis exogenas
        pass

    def fit(self, transactions: List[str]):
        # Fazer fit_mask_, fazer a avaliação de elasticidade com concorrentes/competidores, treina modelo de previsão

        self.skus_, self.mask_ = self.fit_mask_(transactions)

    def predict(self, h, delta_price):
        # Sob modelo de nixtla, faz a previsão fazendo correção da previsão 
        adjusted_forecast = self.mlforecast.predict(h=h) * (1 + (self.cross_elasticity_.mean() * delta_price))
        return adjusted_forecast

In [321]:
cd = CycleDemand()
cd.fit(groceries)

In [323]:
cd.analyze_competitors("coffee")

['UHT-milk',
 'butter',
 'butter milk',
 'candles',
 'candy',
 'canned fish',
 'cat food',
 'chicken',
 'chocolate',
 'chocolate marshmallow',
 'citrus fruit',
 'cling film/bags',
 'condensed milk',
 'cream cheese',
 'dental care',
 'dish cleaner',
 'domestic eggs',
 'female sanitary products',
 'fish',
 'flour',
 'frankfurter',
 'frozen dessert',
 'frozen vegetables',
 'fruit/vegetable juice',
 'hamburger meat',
 'hygiene articles',
 'jam',
 'kitchen towels',
 'light bulbs',
 'liquor (appetizer)',
 'long life bakery product',
 'margarine',
 'mayonnaise',
 'meat',
 'mustard',
 'napkins',
 'oil',
 'pasta',
 'pastry',
 'pickled vegetables',
 'pip fruit',
 'red/blush wine',
 'rice',
 'roll products',
 'salty snack',
 'sausage',
 'seasonal products',
 'shopping bags',
 'sliced cheese',
 'soft cheese',
 'soups',
 'sparkling wine',
 'specialty vegetables',
 'sugar',
 'sweet spreads',
 'syrup',
 'waffles',
 'whipped/sour cream',
 'whole milk',
 'yogurt']